## Введение в регрессионный анализ

*Алла Тамбовцева*

### Практикум 7. Критерий Стьюдента для двух выборок

Импортируем всё необходимое:

* библиотеку `pandas`
* модуль `formula.api` из библиотеки `statsmodels`
* модуль `stats` из библиотеки `scipy`

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

### Подготовка данных

Загрузим данные с прошлого семинара (файл в папке по [ссылке](https://disk.yandex.ru/d/sAbsQ59nMou3ZA)) и посмотрим на первые пять строк таблицы:

In [2]:
df = pd.read_stata("aidpers.dta")
df.head()

,countrycode,country,year,period,periodstart,periodend,code,code2,democracy,polity2,...,trade_gdp,L_PC_PRODUCTION,coldwar,invest4ave,govtc4ave,egypt,lgdp2t,demhistory,begindem,icrg4ave
0,AFG,Afghanistan,1961.0,-1,1958,1961,AFG-1,AFG58,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,0,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,1965.0,0,1962,1965,AFG0,AFG62,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,0,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,1969.0,1,1966,1969,AFG1,AFG66,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,0,NaN,NaN,NaN,NaN
3,AFG,Afghanistan,1973.0,2,1970,1973,AFG2,AFG70,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,0,NaN,NaN,NaN,NaN
4,AFG,Afghanistan,1977.0,3,1974,1977,AFG3,AFG74,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,0,NaN,NaN,NaN,NaN


Напоминание про данные и контекст: в исследовании Д.Райта *Aid Effectiveness and the Politics of Personalism* сравнивается эффект иностранной помощи (переменная `l1234natpc`) на экономический рост (переменная `gdpg`) в государствах с низким и высоким уровнем политического персонализма (переменная `personalism`).

Запросим размерность таблицы – число строк и столбцов, вызовем атрибут `.shape`:

In [3]:
# 1199 строк и 112 столбцов
print(df.shape)

(1199, 112)


**NB.** Так как одна строка в таблице – это страна в конкретный год, нормально, что здесь так много строк, это не 1199 стран, это 1199 наблюдений *страна-год*.

Уже по первым строкам в `df` видно, что в данных много пропусков (ячейки `NaN` от *not a number*). Если удалим все строки, где есть хотя бы один пропуск, рискуем резко сократить объём данных. Согласимся на более аккуратный вариант – так как в исследовании речь идет о демократиях и уровне политического персонализма в них, удалим только те строки, где пропущены значения в столбцах `democracy` и `personalism`.

Воспользуемся методом `.dropna()` и укажем в аргументе `subset` столбцы, где нужно искать пропуски:

In [4]:
df.dropna(subset = ["democracy", "personalism"], inplace = True)
print(df.shape) # число строк резко уменьшилось

(287, 112)


> Аргумент `inplace = True` нужен для того, чтобы сохранить изменения в самом `df`, так как по умолчанию метод `.dropna()` возвращает измененную копию датафрейма с удаленными строками, а оригинал `df` не трогает. Если аргумент `inplace` кажется непонятным, можно с таким же успехом пересохранить `df` через обычное `=`, то есть `df = df.dropna(subset = ["democracy", "personalism"])`.

Посмотрим на таблицу частот для столбца `democracy` и убедимся, что изучать автократии на этих данных нецелесообразно:

In [5]:
# 6 не-демократий
df["democracy"].value_counts()

democracy
1.0    281
0.0      6
Name: count, dtype: int64

> Любопытное замечание. Почему выше вместо 0 и 1, целых значений, которыми обычно кодируют качественные данные, стоят 0.0 и 1.0? Проблема в том, что из наличия пропусков в изначальном `df` тип столбца `democracy` автоматически превратился во `float` (на числовых столбцах это работает так). Пропуски мы удалили, а тип столбца остался прежним. В принципе, это ничему не мешает, но если очень хочется, можно пересохранить столбец, переделав его тип в целочисленный через `.astype(int)`.

Вслед за автором поделим наблюдения на группы – страны с высоким уровнем политического персонализма (значения индекса `personalism` более 5.2) и страны с низким уровнем политического персонализма. Добавим бинарный столбец `group`, где 1 соответствует высокому уровню персонализма:

In [6]:
df["group"] = (df["personalism"] > 5.2).astype("int")
df["group"].value_counts()

group
1    145
0    142
Name: count, dtype: int64

Итого: в данных 142 наблюдения для стран с низким уровнем персонализма и 145 наблюдений для стран с высоким уровнем персонализма. 

### Часть 1: сравниваем средние с помощью критерия Стьюдента

Сгруппируем строки по столбцу `group`, извлечем из каждой группы столбец `gdpg` и запросим для него описательные статистики:

In [7]:
df.groupby("group")["gdpg"].describe()

,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
0,139.0,1.035320,2.858935,-7.941399,-0.724125,1.116417,2.880126,8.591186
1,142.0,1.653199,3.132408,-7.607378,-0.189485,1.935748,3.484507,11.979470


Итак, у нас есть описательные статистики для изменения уровня ВВП стран с низким и высоким уровнем персонализма. Посмотрим на `count` – число заполненных ячеек в столбце `gpdg` в каждой группе. Значения не совпадают с тем, что мы видели выше (здесь 139 наблюдений вместо 142 в группе 0 и 142 вместо 145 в группе 1). О чем это говорит? Что в столбце `gdpg` есть пропуски. Это стоит учесть в дальнейшем, поскольку многие функции для реализации статистических критериев по умолчанию не работают при наличии пропусков.

Сравним средние. В странах с низким уровнем персонализма среднее изменение уровня ВВП составляет 1.03, в странах с высоким уровнем персонализма – 1.65. Велика ли разница в 0.62 единицы? С одной стороны, это не совсем 0. С другой стороны, без дополнительной информации о распределении данных непонятно. Посмотрим на диапазон значений – обратим внимание на минимум и максимум по каждой группе. В первой группе значения `gdpg` варьируются от -7.94 до 8.59, во второй – от 7.61 до 11.98. При таком широком диапазоне значений разница в 0.62 кажется несущественной. Но мы это проверим статистически!

Чтобы статистически проверить, отличаются ли средние в двух независимых выборках (или отлична ли разность средних от нуля), нам понадобится двухвыборочный критерий Стьюдента. В случае независимых выборок возможно два варианта этого критерия: в предположении равенства дисперсий двух генеральных совокупностей или без этого предположения. На лекции рассматривался только первый вариант, его и возьмем. И тут мы даже не исказим действительность – стандартные отклонения в двух группах отличаются несильно (2.86 против 3.13), а значит, никто не мешает предположить равенство дисперсий генеральных совокупностей.

Прежде чем переходить к проверке гипотезы о равенстве средних, убедимся, что вместо `.describe()` можно было указать методы `.mean()` и `.std()`:

In [8]:
# средние
df.groupby("group")["gdpg"].mean()

group
0    1.035320
1    1.653199
Name: gdpg, dtype: float32

In [9]:
# стандартные отклонения
df.groupby("group")["gdpg"].std()

group
0    2.858935
1    3.132408
Name: gdpg, dtype: float32

И вспомним код с лекции (из слайдов) – то же, но без `.groupby()`:

In [10]:
# выбираем строки через фильтр на group, из полученной
# после фильтрации таблицы выбираем столбец gdpg
# и по нему вычисляем среднее

print(df[df["group"] == 0]["gdpg"].mean())
print(df[df["group"] == 1]["gdpg"].mean())

1.0353204
1.653199


In [11]:
# выбираем строки через фильтр на group, из полученной
# после фильтрации таблицы выбираем столбец gdpg
# и по нему вычисляем ст отклонение

print(df[df["group"] == 0]["gdpg"].std())
print(df[df["group"] == 1]["gdpg"].std())

2.8589346
3.1324077


Для реализации критерия Стьюдента для двух независимых выборок нам понадобится функция 
`ttest_ind()` из модуля `stats` библиотеки `scipy` (от *t-test for independent samples*). На вход она принимает две выборки – два набора значений. В нашем случае это значения столбца `gdpg` в таблице для группы 0 и для группы 1:

In [12]:
stats.ttest_ind(df[df["group"] == 0]["gdpg"], 
                df[df["group"] == 1]["gdpg"])

TtestResult(statistic=np.float32(nan), pvalue=np.float32(nan), df=np.float32(nan))

Что-то пошло не так! Результаты вроде есть (наблюдаемое значение статистики, p-value и число степеней свободы), но они пустые. Проблема кроется в наличии пропусков в данных. Если есть хотя бы один `NaN`, функция не вычисляет средние и дисперсии, и, следовательно, все остальное. Чтобы это скорректировать, добавим аргумент `nan_policy = "omit"`, который устанавливает игнорирование пропусков:

In [13]:
stats.ttest_ind(df[df["group"] == 0]["gdpg"], 
                df[df["group"] == 1]["gdpg"], 
                nan_policy = "omit")

TtestResult(statistic=np.float32(-1.726008), pvalue=np.float32(0.085453056), df=np.float32(279.0))

Теперь все хорошо:

* наблюдаемое значение статистики критерия равно $t_{набл} = -1.72$
* p-value = 0.085
* число степеней свободы 279

На уровне значимости 5% можем сделать вывод, что гипотеза $H_0: E(Y_0) = E(Y_1)$ не отвергается (p-value больше 0.05, наблюдаемое значение статистики по модулю не больше 2), а значит, средние изменения ВВП в странах с низким и высоким уровнем политического персонализма не различаются. 

Обратим внимание на число степеней свободы, в таких задачах оно вычисляется как $\text{df} = n_1 + n_2 - 2$, где $n_1$ и $n_2$ – объемы выборок. Объемы выборок в нашем случае – число заполненных ячеек в `gdpg` в группах 0 и 1, значения, которые мы видели в `count` в выдаче `.groupby()`, это 139 и 142. Проверим:

In [14]:
139 + 142 - 2 # верно!

279

Небольшое замечание: у функции `ttest_ind()` есть еще аргумент `equal_var`, который равен `True` или `False` в зависимости от варианта критерия (предполагается ли равенство дисперсий генеральных совокупностей или нет). Если мы рассматриваем вариант с равными дисперсиями , добавлять его необязательно, он и так равен `True` по умолчанию. Можем добавить и убедиться, что ничего не изменилось:

In [15]:
stats.ttest_ind(df[df["group"] == 0]["gdpg"], 
                df[df["group"] == 1]["gdpg"], 
                equal_var = True, 
                nan_policy = "omit")

TtestResult(statistic=np.float32(-1.726008), pvalue=np.float32(0.085453056), df=np.float32(279.0))

### Часть 2: сравниваем средние с помощью линейной модели

Теперь проделаем то же самое – сравним средние в двух группах – с помощью линейной модели. Такое мы уже делали, для решения этой задачи нам понадобится модель, где зависимой переменной выступает экономический рост (`gdpg`), а независимой – бинарный признак для группы (`group`). 

Оценим линейную модель и запросим выдачу с результатами:

In [16]:
mod = smf.ols("gdpg ~ group", data=df).fit()
print(mod.summary())

                            OLS Regression Results                            
Dep. Variable:                   gdpg   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     2.979
Date:                Thu, 26 Nov 2026   Prob (F-statistic):             0.0855
Time:                        01:19:13   Log-Likelihood:                -706.45
No. Observations:                 281   AIC:                             1417.
Df Residuals:                     279   BIC:                             1424.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.0353      0.254      4.068      0.0

Запишем уравнение модели:

$$
\widehat{gdpg}_i = 1.035 + 0.617 \times group_i
$$

Как мы уже знаем, константа `Intercept` = 1.035 – это среднее значение зависимой переменной, если `group` равно 0, а оценка коэффициента при `group` – это разница в средних двух групп. Если сравнить с описательными статистиками по группам, полученными ранее, все так и есть:

In [17]:
mean0 = df[df["group"] == 0]["gdpg"].mean()
mean1 = df[df["group"] == 1]["gdpg"].mean()
print("Среднее в группе 0:", mean0)
print("Разница средних:", mean1 - mean0)

Среднее в группе 0: 1.0353204
Разница средних: 0.61787856


Результаты в столбце `t` и `P>|t|` также совпадают с тем, что мы получили ранее при реализации двухвыборочного критерия Стьюдента. Единственное, что не совпадает – знак у наблюдаемого значения t-статистики. Но это некритично и довольно предсказуемо. В функции `ttest_ind()` ранее на первом месте мы указали выборку для группы 0, поэтому при вычислении статистики из среднего для стран с низким уровнем персонализма вычиталось среднее по странам с высоким уровнем персонализма. А при построении модели оценка коэффициента при бинарной переменной получается вычитанием из среднего по группе, закодированной 1, среднее по группе, закодированное 0.

**Дополнение.** Несколько полезных фактов на будущее для желающих.

1. Если бы нас интересовало сравнение трех и более групп вместо двух, вместо критерия Стьюдента мы бы использовали ANOVA. Этот статистический метод, как и критерий Стьюдента, предполагает, что выборки взяты из нормального распределения. На практике при достаточно больших выборках формальной проверкой на нормальность пренебрегают, однако соответствующие статистические критерии существуют. В Python часто используют критерий Шапиро-Уилка, в `stats` это функция [shapiro()](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html).

2. Если бы мы в качестве инструмента использовали только двухвыборочный критерий Стьюдента, в случае трех и более групп его пришлось бы применять попарно. Например, первую группу сравнить со второй, первую – с третьей, и вторую – с третьей. Технически этот подход имеет право на жизнь, но тогда стоит учесть, что полученные по итогам тестов p-value нужно сравнивать не с исходным уровнем значимости $\alpha$, а с $\alpha/k$, где $k$ – количество сравнений (число пар, не число групп). Эта корректировка называется [поправкой Бонферрони](https://ru.wikipedia.org/wiki/%D0%9F%D0%BE%D0%BF%D1%80%D0%B0%D0%B2%D0%BA%D0%B0_%D0%BD%D0%B0_%D0%BC%D0%BD%D0%BE%D0%B6%D0%B5%D1%81%D1%82%D0%B2%D0%B5%D0%BD%D0%BD%D1%83%D1%8E_%D0%BF%D1%80%D0%BE%D0%B2%D0%B5%D1%80%D0%BA%D1%83_%D0%B3%D0%B8%D0%BF%D0%BE%D1%82%D0%B5%D0%B7), она используется при множественных сравнениях.